<center><h1>CSCI 444: HW 1</h1></center>
<br>
<br>
Name: Sushil Dalavi
<br>
Email ID: sdalavi@usc.edu
<br>
USC ID: 9574285313

Overall guidelines:
- Only change the cells which say: ```# INSERT CODE/TEXT HERE```
- To actually learn how to write code, avoid the use of generative AI. Trust me, it's not going to help you in the long run :)

# Q1. Probability (1)

Say we have three random variables $A$ and $B$ and $C$. Note that we’re using standard probability
theory notation where $P(A, B) = P(B, A)$, which simply means the joint probability of both A
and B occurring.

### Q1.1
(0.5 point)

Which of the following statements is always true?
1. $P(A|B) = P(B|A)$
2. $P(A, B) = P(A) + P(B)$
3. $P(A|B) = max(P(A),P(B))$
4. $P(A, B) = P(B|A)P(A)$

5. $P(A, B, C) = P(A)P(B)P(C)$
6. $P(A, B, C) = P(A)P(C)$
7. $P(A, B, C) = P(A)P(B|A)P(C|A, B)$
8. $P(A)=\sum_{b ∈ \text{domain}(B)} P(A, B=b)$
9. $P(A)=\sum_{b ∈ \text{domain}(B)} P(A|B=b)P(B=b)$
10. $\log(1/P(A)) = 1 - log P(A)$

### Q1.2
(0.5 point)

Now assume that A, B, and C are all independent of each other. Which of the above statements
are now true?

**ANSWERS FOR THE ABOVE SUBPARTS:-**

#### Q1.1
The statements that are **always true** (i.e., true under all circumstances, without assuming independence) are:

**4.** $P(A, B) = P(B|A)P(A)$  
This follows directly from the definition of conditional probability.

**7.** $P(A, B, C) = P(A)P(B|A)P(C|A, B)$ 
This is the chain rule of probability and holds for any joint distribution.

**8.** $P(A)=\sum_{b ∈ \text{domain}(B)} P(A, B=b)$  
This is marginalization: the probability of \(A\) is obtained by summing the joint probability over all possible values of \(B\).

**9.** $P(A)=\sum_{b ∈ \text{domain}(B)} P(A|B=b)P(B=b)$  
This is the law of total probability and follows directly from marginalization.

Therefore the statements which are always true are: 4, 7 ,8 and 9.

#### Q1.2


Now assume that \(A\), \(B\), and \(C\) are **mutually independent**. Under independence, the occurrence of one variable provides no information about the others. Formally, this implies:
\[
$P(A, B, C) = P(A)P(B)P(C)$
\]
and
\[
$P(B|A)=P(B)$, $P(C|A,B)=P(C)$.
\]

The statements that are true under this assumption are:

**4.** $P(A, B) = P(B|A)P(A)$ 
This identity is always true by the definition of conditional probability, regardless of independence.

**5.** $P(A, B, C) = P(A)P(B)P(C)$  
This statement becomes true due to the independence assumption, which allows the joint probability to factor into the product of marginal probabilities.

**7.** $P(A, B, C) = P(A)P(B|A)P(C|A, B)$  
This is the chain rule of probability and is always valid. Under independence, it simplifies to statement (5).

**8.** $P(A)=\sum_{b ∈ \text{domain}(B)} P(A, B=b)$  
This is marginalization and remains true regardless of independence.

**9.** $P(A)=\sum_{b ∈ \text{domain}(B)} P(A|B=b)P(B=b)$ 
This is the law of total probability and also remains true under independence.

Therefore, the statements which are true under the independence assumption are: **4, 5, 7, 8, and 9.**

# Q2. n-gram language model (3)

Create a 5-gram language model trained on the Shakespeare dataset. Store the probabilities P(word|context) in the relevant python dictionaries. Do not use any smoothing (until Question 4). Pay special attention to beginning and end of sequences.

In [1]:
import requests
import collections
import random
import sys
import numpy as np
from tqdm.notebook import tqdm
import math

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
response.raise_for_status() # Raise an exception for invalid HTTP status codes
text_data = response.text
print(f"Length of text: {len(text_data)} \nFirst 100 characters:\n---\n{text_data[:100]}\n---")

Length of text: 1115394 
First 100 characters:
---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
---


In [3]:
# sample
pos = random.randint(0, len(text_data) - 1000)
print(f"Randomly selected 100 characters:\n---\n{text_data[pos:pos+100]}\n---")

Randomly selected 100 characters:
---
ur choice is not so rich in worth as beauty,
That you might well enjoy her.

FLORIZEL:
Dear, look up
---


In [4]:
# preprocessing - do not change
text_data = response.text
text_data = text_data.replace(',',' , ').replace('.',' . ').replace('?',' ? ').replace('!',' ! ')
text_data = text_data.replace('  ', ' ')
text_data = text_data.replace('\n\n','\n').replace('\n',' </s> <s> ')
text_data = '<s> ' + text_data + ' </s>'
print(f"Length of preprocessed text: {len(text_data)} \nFirst 100 characters, preprocessed:\n---\n{text_data[:100]}\n---")

Length of preprocessed text: 1451204 
First 100 characters, preprocessed:
---
<s> First Citizen: </s> <s> Before we proceed any further , hear me speak .  </s> <s> All: </s> <s> 
---


In [5]:
test_data = text_data[-10_000:] # Remember, test data must NOT overlap with train data
text_data = text_data[:-10_000]
len(text_data), len(test_data)

(1441204, 10000)

In [6]:
# split words
data = text_data.split(' ')
len(data), data[:10]

(314074,
 ['<s>',
  'First',
  'Citizen:',
  '</s>',
  '<s>',
  'Before',
  'we',
  'proceed',
  'any',
  'further'])

In [7]:
# UPDATE: Removing unseen tokens from test_data
vocab = set(data)
test_data = ' '.join([_ for _ in test_data.split(' ') if _ in vocab])
print(test_data[:100])

one , or little .  </s> <s> GONZALO: </s> <s> How and lusty the grass looks ! how green !  </s> <s> 


Now, write code to update the n-gram frequencies in the corpus.
For example, five[('a','b','c', 'd', 'e')] should return $P('e' | a','b','c', 'd')$

In [8]:
five = collections.defaultdict(lambda:0)
four = collections.defaultdict(lambda:0)
tri = collections.defaultdict(lambda:0)
bi = collections.defaultdict(lambda:0)
uni = collections.defaultdict(lambda:0)

# INSERTED CODE HERE
uni_counts = collections.Counter()
bi_counts = collections.Counter()
tri_counts = collections.Counter()
four_counts = collections.Counter()
five_counts = collections.Counter()

# Count contexts (denominators)
ctx1_counts = collections.Counter()   # count(w1) for bigram denom
ctx2_counts = collections.Counter()   # count(w1,w2) for trigram denom
ctx3_counts = collections.Counter()   # count(w1,w2,w3) for 4-gram denom
ctx4_counts = collections.Counter()   # count(w1,w2,w3,w4) for 5-gram denom

# Unigrams
for w in data:
    uni_counts[w] += 1

# Bigrams + context counts
for i in range(len(data) - 1):
    w1, w2 = data[i], data[i+1]
    bi_counts[(w1, w2)] += 1
    ctx1_counts[w1] += 1

# Trigrams + context counts
for i in range(len(data) - 2):
    w1, w2, w3 = data[i], data[i+1], data[i+2]
    tri_counts[(w1, w2, w3)] += 1
    ctx2_counts[(w1, w2)] += 1

# 4-grams + context counts
for i in range(len(data) - 3):
    w1, w2, w3, w4 = data[i], data[i+1], data[i+2], data[i+3]
    four_counts[(w1, w2, w3, w4)] += 1
    ctx3_counts[(w1, w2, w3)] += 1

# 5-grams + context counts
for i in range(len(data) - 4):
    w1, w2, w3, w4, w5 = data[i], data[i+1], data[i+2], data[i+3], data[i+4]
    five_counts[(w1, w2, w3, w4, w5)] += 1
    ctx4_counts[(w1, w2, w3, w4)] += 1

# Convert counts -> probabilities P(word | context)
total_tokens = sum(uni_counts.values())

for w, c in uni_counts.items():
    uni[w] = round(c / total_tokens, 5)

# bigram P(w2|w1)
for (w1, w2), c in bi_counts.items():
    bi[(w1, w2)] = round(c / ctx1_counts[w1], 3)

# trigram P(w3|w1,w2)
for (w1, w2, w3), c in tri_counts.items():
    tri[(w1, w2, w3)] = round(c / ctx2_counts[(w1, w2)], 3)

# 4-gram P(w4|w1,w2,w3)
for (w1, w2, w3, w4), c in four_counts.items():
    four[(w1, w2, w3, w4)] = round(c / ctx3_counts[(w1, w2, w3)], 3)

# 5-gram P(w5|w1,w2,w3,w4)
for (w1, w2, w3, w4, w5), c in five_counts.items():
    five[(w1, w2, w3, w4, w5)] = round(c / ctx4_counts[(w1, w2, w3, w4)], 3)



In [9]:
# Evaluation - Do not change
assert five[('Before','we','proceed','any','further')] == 1.0 # prob of last given prev 4
assert four[('<s>','First','Citizen:','</s>')] == 1.0 # prob of last given prev 3
assert tri[('<s>','First','Citizen:')] == 0.172 # prob of last given prev 2
assert round(bi[('First','Citizen:')],3) == 0.169 # prob of last given prev 1
assert round(uni[('Citizen:')],5) == 0.00031 # prob of last

Using the dictionaries containing unigram, bigram, ... five-gram probabilities, return probability P(word|context) as a single scalar. Here context is a list of previous tokens, e.g., [`You`, `may`, `do`, `as`, `you`] and word is a single potential next word, e.g., `like`.

In [10]:
def get_prob_from_lm(context=[], word=""):
  # INSERTED CODE HERE
  ctx = list(context)[-4:]  # use last 4 tokens at most

    # 5-gram
  if len(ctx) >= 4:
        key = (ctx[-4], ctx[-3], ctx[-2], ctx[-1], word)
        if key in five:
            return five[key]

    # 4-gram
  if len(ctx) >= 3:
        key = (ctx[-3], ctx[-2], ctx[-1], word)
        if key in four:
            return four[key]

    # 3-gram
  if len(ctx) >= 2:
        key = (ctx[-2], ctx[-1], word)
        if key in tri:
            return tri[key]

    # 2-gram
  if len(ctx) >= 1:
        key = (ctx[-1], word)
        if key in bi:
            return bi[key]

    # unigram (stored as string keys)
  if word in uni:
        return uni[word]

  return 0.0


In [11]:
get_prob_from_lm(["<s>", "First"], "Citizen:")

0.172

# Q3. Evaluate Perplexity (2.5)

Now let's evaluate the models quantitively using the intrinsic metric **perplexity**.

Recall perplexity is the inverse probability of the test text
$$\text{ppl}(w_1, \dots, w_n) = p(w_1, \dots, w_n)^{-\frac{1}{N}}$$

For an n-gram model, perplexity is computed by
$$\text{ppl}(w_1, \dots, w_n) = (\prod_i p(w_{i+n}|w_i^{i+n-1})^{-\frac{1}{N}}$$

To get rid of numerical issue, we usually compute through:
$$\text{ppl}(w_1, \dots, w_n) = \exp(-\frac{1}{N}\sum_i \log p(w_{i+n}|w_i^{i+n-1}))$$


*Input:*

+ **prob_function**: the `get_prob_from_lm` you implemented in Q2.
+ **data**: test data is a string of text containing multiple words.

*Output:*
+ the perplexity of given data under a language model defined by **prob_function**

Note that you do NOT need to optimize the language model in any way so as to minimize perplexity. Your reported perplexity will have no correlation with your score on this assignment, as far as it is implemented correctly.

In [12]:
def compute_perplexity(prob_function=get_prob_from_lm, data=test_data):
  
  # Hint: something = prob_function(context, word)
  # INSERTED CODE HERE
  eps = 1e-12  # avoid log(0)
  tokens = data.split(' ')
  N = len(tokens)
  if N == 0:
      return float('inf')

  log_sum = 0.0

  for i in range(N):
      # use up to last 4 previous tokens as context (for 5-gram)
      context = tokens[max(0, i-4):i]
      word = tokens[i]

      p = prob_function(context, word)

      # if unseen -> p = 0, avoid log(0)
      if p <= 0:
          p = eps

      log_sum += math.log(p)

  ppl = math.exp(-log_sum / N)
  return ppl

In [13]:
simple_data = 'Before we proceed any further , hear me speak .'
compute_perplexity(data=simple_data)

3.6903913086054048

Can the perplexity of a sentence under any 5-gram model be 1? Explain.

**Answer** 
Yes. Perplexity can be 1 if the model assigns probability 1 to every next-token in the sentence given its context (i.e., each context deterministically predicts the next word). In that case the average negative log probability is 0, so perplexity = exp(0) = 1. This can happen in a degenerate/deterministic language model or if all contexts in the sentence have exactly one continuation with probability 1.

In [14]:
compute_perplexity(data=test_data)

126.85207896508824

**Explain** the resulting perplexity above on test data.
Conjecture (in text, not code) how it could be improved?

**Answer:**
The perplexity on the test data is relatively high because the 5-gram language model suffers from severe data sparsity. Many word sequences in the held-out test set do not appear in the training data, especially for higher-order n-grams. Since this model uses no smoothing, unseen n-grams are assigned zero probability, causing the model to frequently back off to lower-order n-grams or assign extremely small probabilities, which increases the average negative log probability and thus the perplexity.
The perplexity could be improved by applying smoothing techniques such as interpolation or Kneser–Ney smoothing, which distribute probability mass across lower-order n-grams. Additionally, using a smaller n to reduce sparsity, training on more data, and handling rare or unseen words with an <unk> token would also help reduce perplexity.

# Q4. Interpolation Smoothing (1)

Using the dictionaries containing unigram, bigram, ... five-gram probabilities, return probability P(word|context) where context is the previous string, e.g., `You may do as you` and word is a single potential next word, e.g., `like`.

Unlike in Q2, this time use interpolation smoothing as discussed in class.

In [15]:
l1=0.2; l2=0.2; l3=0.2; l4=0.2; l5=0.2
# lambdas needed for Interpolation - you do NOT need to change these for the assignment
def get_prob_from_lm_with_interpolation(context=[], word=""):
    # INSERTED CODE HERE
  ctx = list(context)[-4:]  # use last 4 tokens max

  # unigram (stored as string keys)
  p1 = uni[word] if word in uni else 0.0

  # bigram
  p2 = 0.0
  if len(ctx) >= 1:
      p2 = bi.get((ctx[-1], word), 0.0)

  # trigram
  p3 = 0.0
  if len(ctx) >= 2:
      p3 = tri.get((ctx[-2], ctx[-1], word), 0.0)

  # 4-gram
  p4 = 0.0
  if len(ctx) >= 3:
      p4 = four.get((ctx[-3], ctx[-2], ctx[-1], word), 0.0)

  # 5-gram
  p5 = 0.0
  if len(ctx) >= 4:
      p5 = five.get((ctx[-4], ctx[-3], ctx[-2], ctx[-1], word), 0.0)

  return l1*p1 + l2*p2 + l3*p3 + l4*p4 + l5*p5
get_prob_from_lm_with_interpolation(['Before', 'we', 'proceed', 'any'],
                                    'further')

0.6032200000000001

Now we re-evaluate perplexity with the new probability estimates (interpolated).

In [16]:
compute_perplexity(prob_function=get_prob_from_lm_with_interpolation,
                   data=simple_data)

9.820019419319822

In [17]:
compute_perplexity(prob_function=get_prob_from_lm_with_interpolation,
                   data=test_data)

173.3835302780666

# Q5. Text Generation (2.5)

Our final goal is to generate/infer/sample from the language model that we have created.
Given a prefix or prompt, the model can generate text according to estimated next word probabilities (similar to any LLM, e.g., GPT).

Complete the following function using the dictionaries already available to sample (you may use `random.choice` function) from the distribution of possible next tokens. Remember to randomly sample (from a weighted distribution over all possible next words in the vocabulary) instead of picking only the most likely next word.

Input:
- dictionaries `uni`, `bi`, `tri`, `four`, `five`
- **n_generate**: number of maximum words to infer. E.g., `2`
- **prefix**: the context on which to condition text generation. E.g., `as you`

Output: string output containing prefix and generated words separated by whitespace. E.g., `as you like it`

In [18]:
prefix = 'as you say you' # "hear me"
def generate(prob_fn=get_prob_from_lm_with_interpolation,
             prefix=prefix, n_generate=20):
    # INSERT CODE HERE
    tokens = prefix.split(' ')
    vocab_list = list(vocab)  # vocab is from earlier: set(data)

    # prevent trivial immediate termination
    min_len_before_end = 5

    for _ in range(n_generate):
        context = tokens[-4:]  # 5-gram uses last 4 tokens as context

        weights = []
        for w in vocab_list:
            # don't generate <s> inside a sentence
            if w == '<s>':
                weights.append(0.0)
                continue

            # don't allow </s> too early (otherwise generation ends immediately)
            generated_so_far = len(tokens) - len(prefix.split(' '))
            if w == '</s>' and generated_so_far < min_len_before_end:
                weights.append(0.0)
                continue

            weights.append(prob_fn(context, w))

        total = sum(weights)

        # if everything is zero, fall back to random choice
        if total == 0:
            next_word = random.choice(vocab_list)
        else:
            weights = [p / total for p in weights]
            next_word = random.choices(vocab_list, weights=weights, k=1)[0]

        tokens.append(next_word)

        if next_word == '</s>':
            break

    return ' '.join(tokens)

In [19]:
generate(prob_fn=get_prob_from_lm)

'as you say you are no soldiers; a malcontent Justice , O royal duke my I fashion this kept waking and with us but'

In [20]:
generate(prob_fn=get_prob_from_lm_with_interpolation)

'as you say you would not this you , sir .  his he Why , then , make their pastime at my troth'

Compare the outputs of the two language models. Why do you think a difference exists?

Answer:- The two models continue the same prefix in noticeably different ways because they score the next word differently. The backoff model mostly trusts the highest-order n-gram it can find, and when that exact context isn’t in the training data it drops to shorter contexts. That can make it jump around more and it can also end quickly if it samples </s>.
The interpolated model blends 1-gram through 5-gram probabilities, so it usually has a smoother, less “all-or-nothing” distribution. Because the probability distribution changes (and we are sampling randomly), the continuation you get can look quite different even with the same prompt.

# The End
Voila! If you were able to complete the assignment, you have successfully created a language model in barebones Python! You have trained it on a text corpus and can use it to generate arbitrary text just like ChatGPT!